In [2]:
import pandas as pd
from pathlib import Path

In [3]:
pip install pandas psycopg2 -binary sqlalchemy

Note: you may need to restart the kernel to use updated packages.



[optparse.groups]Usage:[/]   
  c:\Users\WHH\OneDrive\Desktop\github_project\venv\Scripts\python.exe -m pip install \[options] <requirement specifier> \[package-index-options] ...
  c:\Users\WHH\OneDrive\Desktop\github_project\venv\Scripts\python.exe -m pip install \[options] -r <requirements file> \[package-index-options] ...
  c:\Users\WHH\OneDrive\Desktop\github_project\venv\Scripts\python.exe -m pip install \[options] [-e] <vcs project url> ...
  c:\Users\WHH\OneDrive\Desktop\github_project\venv\Scripts\python.exe -m pip install \[options] [-e] <local project path> ...
  c:\Users\WHH\OneDrive\Desktop\github_project\venv\Scripts\python.exe -m pip install \[options] <archive url/path> ...

no such option: -b


In [4]:
import os
import pandas as pd
import numpy as np 
import psycopg2
from dotenv import load_dotenv
from sqlalchemy import create_engine

In [5]:
load_dotenv(dotenv_path="../.env")


True

In [6]:
conn = psycopg2.connect(
    host = "***",
    port ="5432",
    database = "neondb",
    user = "neondb_owner",
    password = "***"
)

print ("connected successfully")

connected successfully


# Loading clean sql table into pandas

In [7]:
engine = create_engine("postgresql+psycopg2://neondb_owner:***@***:5432/neondb?sslmode=require")

In [8]:
query = 'select * from student_project."properties_cleaned";'
df= pd.read_sql(query,engine)
print ("Succes ! shape:",df.shape)
display(df.head())

Succes ! shape: (105, 13)


,Property_id,Status,List_date,List_price,Amount_reduced,City,State,Postal_code,Bedrooms,Bathrooms,Property_sqft,Property_type,Price_per_sqft
0,1000618972,for_sale,2026-09-01,759000.0,0.0,Los Angeles,California,90004,2,2.0,1087.0,condos,698.25
1,1010818622,for_sale,2025-11-21,1245000.0,302000.0,Los Angeles,California,90004,4,3.0,3088.0,multi_family,403.17
2,1141459809,for_sale,2026-09-25,3400000.0,0.0,Los Angeles,California,90004,0,32.0,18604.0,multi_family,182.76
3,1141608873,for_sale,2026-09-10,3590000.0,0.0,Los Angeles,California,90004,24,14.0,10884.0,multi_family,329.84
4,1308593528,for_sale,2026-09-01,2050000.0,150000.0,Los Angeles,California,90004,7,12.0,7552.0,multi_family,271.45


In [9]:
print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 105 entries, 0 to 104
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Property_id     105 non-null    int64  
 1   Status          105 non-null    str    
 2   List_date       105 non-null    str    
 3   List_price      105 non-null    float64
 4   Amount_reduced  105 non-null    float64
 5   City            105 non-null    str    
 6   State           105 non-null    str    
 7   Postal_code     105 non-null    int64  
 8   Bedrooms        105 non-null    int64  
 9   Bathrooms       105 non-null    float64
 10  Property_sqft   105 non-null    float64
 11  Property_type   105 non-null    str    
 12  Price_per_sqft  105 non-null    float64
dtypes: float64(5), int64(3), str(5)
memory usage: 10.8 KB
None


In [10]:
print(df.describe())

        Property_id    List_price  Amount_reduced  Postal_code    Bedrooms  \
count  1.050000e+02  1.050000e+02    1.050000e+02        105.0  105.000000   
mean   2.577398e+09  2.567547e+06    2.351595e+05      90004.0    6.000000   
std    1.780217e+09  2.997166e+06    1.930677e+06          0.0    5.480735   
min    1.000619e+09  4.450000e+05    0.000000e+00      90004.0    0.000000   
25%    2.074430e+09  9.990000e+05    0.000000e+00      90004.0    3.000000   
50%    2.157831e+09  1.679000e+06    0.000000e+00      90004.0    4.000000   
75%    2.267468e+09  3.245000e+06    4.900000e+04      90004.0    6.000000   
max    9.933308e+09  1.999500e+07    1.980000e+07      90004.0   24.000000   

        Bathrooms  Property_sqft  Price_per_sqft  
count  105.000000     105.000000      105.000000  
mean     6.333333    4397.457143      691.663810  
std      7.250553    4842.275385      382.366226  
min      1.000000     659.000000      182.760000  
25%      2.000000    1646.000000      444.

1. How do average  property prices vary by property type ?

In [ ]:
df["Property_type"]=df["Property_type"].str.lower().str.replace("-"," ")
Type_ranking=df.groupby("Property_type").agg(Average_price=("List_price","mean"),
                                    toal_listing=("Property_id","count")).reset_index()

Type_ranking =Type_ranking.sort_values(by="Average_price",ascending=False)

Type_ranking    

,Property_type,Average_price,toal_listing
2,single_family,3.256631e+06,45
1,multi_family,2.685949e+06,39
0,condos,8.759474e+05,19
3,townhomes,8.245000e+05,2


2. How can we categorize properties into budget tiers using custom logic ?

In [27]:
df["Price_tier"]=df["List_price"].apply(lambda price: '1. Entry Level' if price < 1000000 else
                                        ('2. Mid_range' if 1000000 <= price <= 2000000 else
                                        ('3. High-end' if 2000001 <= price <= 5000000 else '4. Over $5m (Ultra-Luxury)')))
sorted_subset=df[["Property_id","List_price","Price_tier"]].sort_values(by="Price_tier").head(10)

display(sorted_subset)

,Property_id,List_price,Price_tier
0,1000618972,759000.0,1. Entry Level
71,2230723624,489000.0,1. Entry Level
42,2136659829,999000.0,1. Entry Level
65,2208103197,749000.0,1. Entry Level
63,2202103380,485000.0,1. Entry Level
62,2202028848,445000.0,1. Entry Level
59,2190588204,900000.0,1. Entry Level
72,2230958805,887000.0,1. Entry Level
31,2085870905,999000.0,1. Entry Level
53,2170539126,775000.0,1. Entry Level


3. How can we sort and find the top most expensive properties ?

In [13]:
top_properties=df.sort_values(by="List_price",ascending=False).head(5)
print("Top most expensive properties")
display(top_properties[["Property_id","City","List_price","Price_tier"]])

Top most expensive properties


,Property_id,City,List_price,Price_tier
22,2045325560,Los Angeles,19995000.0,4. Over $5m (Ultra-Luxury)
23,2066418083,Los Angeles,19995000.0,4. Over $5m (Ultra-Luxury)
15,1993302304,Los Angeles,9995000.0,4. Over $5m (Ultra-Luxury)
60,2191557118,Los Angeles,7950000.0,4. Over $5m (Ultra-Luxury)
76,2255027927,Los Angeles,7500000.0,4. Over $5m (Ultra-Luxury)


4. How can we format all numerice columns cleanly for executive reporting??

In [16]:
numerice_cols =["List_price","Property_sqft","Price_per_sqft"]
formatted_df= df.copy()
formatted_df[numerice_cols]=formatted_df[numerice_cols].map(
    lambda x :round(x,2)if pd.notnull(x)else x
)
print("Sample of formatted Numeric data")
display(formatted_df[["Property_id","List_price","Property_sqft"]].head(5))

Sample of formatted Numeric data


,Property_id,List_price,Property_sqft
0,1000618972,759000.0,1087.0
1,1010818622,1245000.0,3088.0
2,1141459809,3400000.0,18604.0
3,1141608873,3590000.0,10884.0
4,1308593528,2050000.0,7552.0


How can we merge our property dataset with a regional market rating table?

In [24]:
maintenance_rates={"single_family": 1.2,
                   "multi_family":1.5,
                   "condos": 1.0,
                   "Land":0.8}

reference_data = pd.DataFrame(list(maintenance_rates.items()),columns=["Property_type","Maintenance_Multiplier"])

merge_df= pd.merge(df,reference_data, on ="Property_type",how="left")

merge_df["Adjusted_price"]=merge_df["List_price"]*merge_df["Maintenance_Multiplier"]
print("sucessfully merge with all property_type")
display(merge_df[["Property_id","Property_type","List_price","Maintenance_Multiplier","Adjusted_price"]])

sucessfully merge with all property_type


,Property_id,Property_type,List_price,Maintenance_Multiplier,Adjusted_price
0,1000618972,condos,759000.0,1.0,759000.0
1,1010818622,multi_family,1245000.0,1.5,1867500.0
2,1141459809,multi_family,3400000.0,1.5,5100000.0
3,1141608873,multi_family,3590000.0,1.5,5385000.0
4,1308593528,multi_family,2050000.0,1.5,3075000.0
...,...,...,...,...,...
100,9470287942,multi_family,4495000.0,1.5,6742500.0
101,9500393279,condos,759000.0,1.0,759000.0
102,9735040714,multi_family,1795000.0,1.5,2692500.0
103,9807897530,multi_family,5095000.0,1.5,7642500.0
